# Lab 2: Member and Employer Segmentation

**Time:** about 60 minutes

In Lab 1 you had a target: past audits told you which claims were irregular, and the model learned to
predict that. Now there's no target. The question changes from

> *"What will happen?"*

to

> *"What kinds of members does the Fund actually have?"*

Nobody has labelled members as "type A" or "type B". You'll let an algorithm group similar members
together, and then do the part only a person can do: work out what each group **means** and what the Fund
should do differently for it.

```
member profiles ─► choose features ─► transform & scale ─► k-means ─► profile the clusters ─► name them ─► actions
                                                          └─► hierarchical clustering (a second opinion)
```

## Where this lab fits

**What you've done so far.** On Day 1 you cleaned the Fund's data. In Lab 1 this morning you used past audits to train
a model that predicts which claims are likely to be irregular. That was **supervised** learning: every example came
with the right answer.

**What you'll do now.** You'll use 15,000 member profiles (and, in an exercise, 1,300 employer profiles) to discover
natural groups of members who behave alike. There's no answer to learn from this time. The algorithm finds the
groups; you work out what they mean and what the Fund should do differently for each.

**Why it matters.** The Fund treats all members much the same way: the same letters, the same branch services, the same
reminders. But a 24-year-old who uses the app and a 58-year-old whose employer stopped paying need very different
things. Segments are how member services and compliance can target their limited time.

**What happens next.** Lab 3 returns to the claims model from Lab 1 and tests it far more carefully. The segmentation
ideas come back on Day 5, where finding "unusual" payments with no labels uses a similar way of thinking.

### Theory: unsupervised learning and clustering

**Unsupervised learning** means learning from data that has no target column. Nobody has told the algorithm what
the right answer is, so it can't learn to predict one. Instead it looks for **structure**: patterns in how the rows
relate to each other.

**Clustering** is the most common kind. It puts rows into groups (**clusters**) so that rows in the same group are
similar to each other and different from rows in other groups.

Three things to understand before you start:

1. **"Similar" is something you define**, by choosing which columns go in and how they're scaled. Change the columns,
   and you change the groups.
2. **The algorithm gives numbers, not meanings.** It says "member 1043 is in cluster 2". It never says what cluster 2
   *is*. Naming the groups is your job, and it needs knowledge of the members.
3. **There's no single right answer.** There's no test set and no accuracy. A segmentation is good if it's stable, if
   the groups are clearly different, and above all if the Fund can act on it.

In [ ]:
import os
import warnings

# Skip a slow, noisy CPU-core check on Windows 11 (see Lab 1).
os.environ["LOKY_MAX_CPU_COUNT"] = str(os.cpu_count() - 1)
warnings.filterwarnings("ignore", category=FutureWarning)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster    # hierarchical clustering

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)
SEED = 42

> **Where we are.** The libraries are loaded. Now you'll meet the member profiles: one row per member, summarising
> how they've contributed and contacted the Fund over the last two years.

## 1. The member profiles

Each row is an active member, summarised over the last 24 months.

| Column | Meaning |
|---|---|
| `contribution_months` | Total months of contributions on record |
| `avg_monthly_contribution_tzs` | Average contribution (employer + employee) in months that were posted |
| `posting_regularity_24m` | Share of the last 24 months in which a contribution was actually posted |
| `longest_gap_months_24m` | Longest run of consecutive months with nothing posted |
| `months_since_last_posting` | Months since the last contribution was posted |
| `claims_lodged` | Claims the member has ever lodged |
| `service_contacts_12m` | Calls, visits, emails and messages to the Fund in the last year |
| `digital_share` | Share of those contacts through the app, portal or USSD (empty if no contacts) |
| `statement_requests_12m` | Statements requested in the last year |

In [ ]:
members = pd.read_csv("member_profile.csv", dtype={"member_no": "string"})
print(members.shape)

# .T turns the summary on its side: one row per column, easier to read.
members.describe().T.round(2)

### Theory: distance, the idea everything here rests on

To decide whether two members are similar, the algorithm measures the **distance** between them. Think of each member
as a point on a map where each feature is one direction. With two features you can draw it:

```
posting regularity
   1.0 |   A •        • B
       |
   0.5 |
       |              • C
   0.0 +----------------------- months since last posting
       0       3       6
```

Members A and B both post regularly; C has stopped. The usual measure is the straight-line (**Euclidean**) distance:
square the difference on each feature, add them up, and take the square root. With nine features you can't draw it,
but the arithmetic is the same.

**Why this makes preparation so important.** Each feature contributes its squared difference to the distance. If one
feature's differences are huge numbers (shillings) and another's are tiny (shares between 0 and 1), the big one wins
every time. That's why the next steps transform and scale the columns before clustering.

**The log transform.** `np.log1p(x)` takes the logarithm of `x + 1` (the "+1" avoids the problem of log(0)). It squeezes
large values together: the difference between 1 and 10 months becomes about the same as the difference between 10 and
100. Without it, a few members with very high contributions sit far from everyone else and can end up in a cluster of
their own.

## 2. Choose and prepare the features

Clustering groups rows that are **close** to each other, so two things matter a lot:

1. **Which columns you include.** Every column you add changes what "similar" means. Leave out anything
   that shouldn't drive the grouping. Here, gender and region are left out: you want groups based on how
   members *behave*, and you can check afterwards how gender and region spread across the groups.
2. **Scale.** `avg_monthly_contribution_tzs` is in hundreds of thousands, `posting_regularity_24m` is
   between 0 and 1. Without scaling, the contribution amount would decide everything on its own.

Skewed columns (amounts, counts) get a log transform first, so that a handful of very high earners don't
end up as a cluster of their own.

In [ ]:
# The columns that describe how a member behaves.
FEATURES = [
    "age", "contribution_months", "avg_monthly_contribution_tzs", "posting_regularity_24m",
    "months_since_last_posting", "service_contacts_12m", "digital_share", "statement_requests_12m",
]

# Work on a copy, so the original table stays as it was.
X = members[FEATURES].copy()

# How many gaps does each column have?
X.isna().sum()

In [ ]:
# Fill the gaps. K-means can't handle missing values.

# No contacts at all means no digital share. Assume a typical mix (the median).
X["digital_share"] = X["digital_share"].fillna(X["digital_share"].median())

# Never posted in the last 24 months: treat as 24 months. clip(upper=24) caps the column at 24.
X["months_since_last_posting"] = X["months_since_last_posting"].fillna(24).clip(upper=24)

# No posted months means no average. Use the median.
X["avg_monthly_contribution_tzs"] = X["avg_monthly_contribution_tzs"].fillna(X["avg_monthly_contribution_tzs"].median())

print("gaps left:", X.isna().sum().sum())

In [ ]:
# Log scale for the skewed columns: stops a few very large values dominating.
X["contribution_months"] = np.log1p(X["contribution_months"])
X["avg_monthly_contribution_tzs"] = np.log1p(X["avg_monthly_contribution_tzs"])
X["service_contacts_12m"] = np.log1p(X["service_contacts_12m"])
X["statement_requests_12m"] = np.log1p(X["statement_requests_12m"])

X.head()

### Why scaling matters for clustering

K-means groups members by **distance**. Suppose two members differ by TZS 50,000 in average contribution and by 0.5 in
posting regularity. In raw numbers the first difference is 100,000 times bigger, so the contribution column would
decide every distance on its own. After scaling, each column is measured in "how many standard deviations from the
average member", so a big difference in regularity counts as much as a big difference in contribution.

In [ ]:
# StandardScaler: for every column, subtract its mean and divide by its standard deviation.
scaler = StandardScaler()
Xs = scaler.fit_transform(X)

# Xs is a plain numpy array. Put it in a DataFrame to check: every column now has mean ~0 and std ~1.
Xs_table = pd.DataFrame(Xs, columns=FEATURES)
Xs_table.describe().loc[["mean", "std"]].round(2)

> **Where we are.** The features are chosen, the gaps filled, the skewed columns logged, and every column scaled so
> each one counts equally in the distance. The data is ready to be clustered.

### Theory: how k-means works

K-means finds `k` groups by repeating two simple steps:

1. **Start:** put `k` centre points at random positions among the members.
2. **Assign:** give each member to its nearest centre.
3. **Update:** move each centre to the average position of the members assigned to it.
4. **Repeat** steps 2 and 3 until the centres stop moving.

```
round 1:  centres placed at random  →  members join the nearest one
round 2:  centres move to the middle of their members  →  some members switch
...       until nothing changes
```

Because the start is random, different runs can end in slightly different places. scikit-learn runs the whole thing
several times (`n_init`) and keeps the best, and `random_state` makes the result repeatable.

**How to choose k.** K-means can't tell you how many groups there should be. Two numbers help:

- **Inertia** is the total squared distance from each member to its centre. It always falls as `k` grows (with one
  cluster per member it's zero), so you look for the **elbow**: the point where adding a cluster stops making much
  difference.
- **Silhouette score** compares, for each member, the distance to its own cluster with the distance to the next
  nearest one. It runs from -1 to +1: near +1 means clearly in the right group, near 0 means on the border.

## 3. K-means: how many groups?

K-means needs you to say how many clusters (`k`) you want. There's no single right answer, but two
diagnostics help:

- **Inertia** (the elbow plot): how tightly members sit around their cluster centre. It always falls as `k`
  grows. You look for the point where adding another cluster stops helping much.
- **Silhouette score**: how much closer each member is to its own cluster than to the next nearest one.
  Higher is better. It's slow on large data, so it's computed on a sample.

In [ ]:
# Pick 3,000 random row positions for the silhouette score (it's slow on all 15,000).
rng = np.random.default_rng(SEED)
sample_idx = rng.choice(len(Xs), 3000, replace=False)

# Try k = 2, 3, ... 8 and record both diagnostics for each.
k_values = []
inertias = []
silhouettes = []

for k in range(2, 9):
    km = KMeans(n_clusters=k, n_init=10, random_state=SEED)
    km.fit(Xs)
    k_values.append(k)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(Xs[sample_idx], km.labels_[sample_idx]))
    print(f"k={k}  inertia={km.inertia_:,.0f}  silhouette={silhouettes[-1]:.3f}")

In [ ]:
# The elbow plot.
plt.plot(k_values, inertias, marker="o", color="#2a78d6")
plt.title("Inertia (look for the elbow)")
plt.xlabel("k (number of clusters)")
plt.ylabel("inertia")
plt.show()

In [ ]:
# The silhouette plot.
plt.plot(k_values, silhouettes, marker="o", color="#eb6834")
plt.title("Silhouette (higher is better)")
plt.xlabel("k (number of clusters)")
plt.ylabel("silhouette score")
plt.show()

The diagnostics narrow the choice down. They don't make it for you. The deciding question is a business
one: **can the Fund actually do something different for each group?** Twelve segments that nobody can
remember are worse than four that shape how member services works.

Go with `k = 4` for now. You'll try other values at the end.

In [ ]:
K = 4

# n_init=20: run k-means 20 times from different starting points and keep the best result.
kmeans = KMeans(n_clusters=K, n_init=20, random_state=SEED)
kmeans.fit(Xs)

# labels_ = the cluster number (0 to 3) given to each member.
members["kmeans"] = kmeans.labels_
members["kmeans"].value_counts().sort_index()

> **Where we are.** You've chosen `k = 4` and every member now has a cluster number from 0 to 3. Those numbers are
> just labels the algorithm made up. This section turns them into something a person can understand.

### Theory: profiling clusters

**Profiling** a cluster means describing its typical member. Two views help:

- **Medians in original units** ("the typical member in cluster 2 last posted 9 months ago") are easy to read and
  easy to explain to a branch manager.
- **Centres in scaled units** ("cluster 2 is 1.8 standard deviations above average on months since last posting")
  are easier to compare across features, because every feature is on the same scale. A value near 0 means "like the
  average member"; beyond about ±0.8 means the feature really sets the cluster apart.

A heatmap of the scaled centres shows at a glance which features define each cluster. Name a cluster after the two or
three features that stand out most.

## 4. What does each cluster mean?

The algorithm returns numbers 0 to 3. They mean nothing yet. To give them meaning, compare each cluster's
typical member with everyone else's. Use the **original** units, not the scaled ones, so the table is
readable.

In [ ]:
# The median of each feature, per cluster, in the original units.
profile = members.groupby("kmeans")[FEATURES].median()

# Add the number of members in each cluster.
profile["members"] = members["kmeans"].value_counts().sort_index()
profile.round(2)

Scaled differences are easier to compare across columns. The cluster **centres** are in scaled units: how far each
cluster's average is above (positive) or below (negative) the overall average, in standard deviations.

In [ ]:
# One row per cluster, one column per feature.
centres = pd.DataFrame(kmeans.cluster_centers_, columns=FEATURES)
centres.round(1)

In [ ]:
# The same table as a heatmap: blue = above average, orange = below average.
plt.figure(figsize=(10, 3))
plt.imshow(centres.values, cmap="RdBu", vmin=-2, vmax=2, aspect="auto")
plt.colorbar(fraction=0.02)

# Label the axes with the feature names and cluster numbers.
plt.xticks(range(len(FEATURES)), FEATURES, rotation=30, ha="right")
plt.yticks(range(K), ["cluster 0", "cluster 1", "cluster 2", "cluster 3"])

# Write the value in each square.
for row in range(K):
    for col in range(len(FEATURES)):
        plt.text(col, row, round(centres.values[row, col], 1), ha="center", va="center", fontsize=8)
plt.show()

**Your turn (1).** Stop here and name the clusters before reading on. For each one, write a two-to-four-word name
and one sentence describing the typical member. Use the heatmap: which features make each cluster stand out?

Write your names in the cell below, replacing the placeholders. The rest of the notebook uses whatever you
write.

In [ ]:
# Replace these with your own names after studying the profile table and heatmap.
cluster_names = {
    0: "cluster 0",
    1: "cluster 1",
    2: "cluster 2",
    3: "cluster 3",
}

Remember: **the algorithm didn't invent those names, you did.** K-means only found groups of members whose
numbers look alike. Deciding that a group is "near retirement" or "contributions interrupted" is an analyst's
interpretation, and a different analyst might describe the same group differently. That's why a segmentation
should always be reviewed with the people who serve members every day.

Now look at who is in each group:

In [ ]:
# Give each member their segment name.
members["segment"] = members["kmeans"].map(cluster_names)

# For each segment, the share of its members in each employer sector.
# normalize="index" turns counts into shares that add up to 1 across each row.
sector_mix = pd.crosstab(members["segment"], members["employer_sector"], normalize="index")
sector_mix.T.round(2)

> **Where we are.** Your four clusters have names, and you've checked how many members fall in each. But those groups
> came from one algorithm with one random start. Before anyone builds a member-services plan on them, it's worth
> checking whether a completely different method finds similar groups.

### Theory: hierarchical clustering and linkage

Hierarchical clustering works from the bottom up:

1. Start with every member as a group of one.
2. Find the two closest groups and merge them.
3. Repeat until everything is one group.

The record of every merge is drawn as a **dendrogram**, a tree where the height of each join shows how far apart the two
groups were when they were merged. Cut the tree at a chosen height, and the branches below the cut are your clusters.

**Linkage** is the rule for measuring the distance between two *groups*. **Ward** linkage (used here) merges the pair
of groups whose merger increases the spread within groups the least. That tends to give compact, similar-sized groups,
much like k-means, which makes the comparison fair.

**Why a sample.** The method needs the distance between every pair of members. For n members that's about n² / 2
pairs: 2,000 members give about 2 million pairs, but 15,000 give about 112 million. Memory grows far faster than the
number of rows.

## 5. A second opinion: hierarchical clustering

Hierarchical clustering doesn't need `k` up front. It starts with every member in its own group and keeps
merging the two closest groups until only one is left. The result is a tree (a **dendrogram**), and you decide
where to cut it.

It needs the distance between every pair of members, so memory grows with the *square* of the number of
rows. 15,000 members means about 112 million pairs, which is too many for a laptop. Use a sample of 2,000.

In [ ]:
# The first 2,000 of our random sample.
sample_2000 = sample_idx[:2000]
Xh = Xs[sample_2000]

# Build the tree. method="ward": merge the pair of groups that increases the within-group spread the least.
Z = linkage(Xh, method="ward")

In [ ]:
# Draw only the last 30 merges, otherwise there would be 2,000 leaves.
plt.figure(figsize=(11, 4))
dendrogram(Z, truncate_mode="lastp", p=30, no_labels=True)
plt.title("Dendrogram (last 30 merges)")
plt.ylabel("merge distance")
plt.show()

Long vertical lines mean the two groups being merged were very different. Cutting the tree just below the
longest lines gives you natural groups. Cut it to get 4 clusters and compare with k-means on the same members:

In [ ]:
# Cut the tree so that it gives K groups.
hier_labels = fcluster(Z, t=K, criterion="maxclust")

# The k-means labels of the same 2,000 members.
km_labels = members["kmeans"].values[sample_2000]

# Adjusted Rand index: 1 = the two methods group members identically, 0 = no better than chance.
ari = adjusted_rand_score(km_labels, hier_labels)
print("Adjusted Rand index:", round(ari, 3))

In [ ]:
# Rows = k-means cluster, columns = hierarchical cluster, cells = number of members in both.
pd.crosstab(km_labels, hier_labels, rownames=["k-means"], colnames=["hierarchical"])

> **Where we are.** Two different algorithms broadly agree on the groups, so they probably reflect real differences
> between members and not a quirk of one method. Now comes the part that makes a segmentation worth the effort.

### Theory: what makes a segmentation useful

A segmentation is useful if it passes four tests:

| Test | Question |
|---|---|
| **Distinct** | Are the groups clearly different from each other? |
| **Meaningful** | Can you describe each group in a sentence a branch manager understands? |
| **Big enough** | Is each group large enough to justify its own approach? |
| **Actionable** | Is there something the Fund would do differently for this group, and could it measure whether it worked? |

The last test matters most. A beautifully separated cluster that leads to no different action is a nice chart, not a
result.

If both methods broadly agree, you can be more confident the groups reflect real structure rather than
the quirks of one algorithm. Where they disagree, look at those members. They're often the ones on the
boundary between two segments.

| | K-means | Hierarchical |
|---|---|---|
| Number of groups | You choose `k` first | You choose where to cut the tree afterwards |
| Shape of groups | Round-ish blobs around a centre | Depends on the linkage method |
| Scale | Fine for millions of rows | Memory grows with rows², so thousands, not millions |
| New members | Assign to nearest centre: easy to put into production | No direct way; you'd need a separate classifier |
| Best for | Operational segmentation | Exploring structure and checking k-means |

## 6. From segments to actions

A segmentation is only worth something if it changes what the Fund does. For each segment, fill in a row:

In [ ]:
# Number of members in each segment.
segment_sizes = members["segment"].value_counts()

actions = pd.DataFrame({
    "segment": [cluster_names[0], cluster_names[1], cluster_names[2], cluster_names[3]],
    "members": [segment_sizes[cluster_names[0]], segment_sizes[cluster_names[1]],
                segment_sizes[cluster_names[2]], segment_sizes[cluster_names[3]]],
    "what_they_need": ["", "", "", ""],
    "proposed_action": ["", "", "", ""],
    "channel": ["", "", "", ""],
})
actions

**Your turn (2).** Fill in `what_they_need`, `proposed_action` and `channel` for each segment, drawing on what you
know about how members are served today. Some prompts to get you going:

- Which segment would benefit from an SMS reminder to check their statement?
- Which segment should the compliance team care about, because the problem is really their *employers*?
- Which segment needs pension preparation sessions before they retire?
- Which segment should be nudged towards the app to take pressure off the branches?

## 7. Employers too

**Your turn (3).** Repeat the exercise on `employer_profile.csv`. Suggested features: `headcount_latest`
(log), `avg_wage_tzs` (log), `on_time_rate_24m`, `avg_days_late_24m`, `unpaid_months_24m`,
`partial_payment_rate_24m`, `headcount_cv_24m`. Choose `k`, profile the clusters, and name them. Which
cluster would you hand to the compliance team first?

In [ ]:
# Your code here

## Looking back, looking ahead

**What you did in this lab.** With no target to learn from, you chose behavioural features, logged and scaled them so
each counted fairly in the distance, picked a number of clusters using the elbow and silhouette as guides, profiled and
named the groups, checked them against a second method, and turned each segment into a proposed action.

**The ideas to carry forward.**

- In unsupervised learning, *you* define similarity through the columns and scaling you choose.
- Algorithms find groups; people give them meaning.
- Check a result with a second method before trusting it.
- A segment is only worth having if it changes what the Fund does.

**What happens next.** Lab 3 goes back to supervised learning. It takes the claims models from Lab 1, builds one fair
way to compare them against the review team's real capacity, and checks whether the winner can be trusted.

---

## Answers and discussion

**(1)** Your names will depend on the profile. To check your reading of the heatmap, list for each cluster the
features that sit well above or well below average (more than 0.8 standard deviations away):

In [ ]:
for cluster in range(K):
    centre = centres.loc[cluster]
    high = centre[centre > 0.8].index.tolist()
    low = centre[centre < -0.8].index.tolist()
    print(f"cluster {cluster}:")
    print("    well above average:", high)
    print("    well below average:", low)

For example, a cluster with high `age` and `contribution_months` is a candidate for "near retirement"; one with
high `months_since_last_posting` and low `posting_regularity_24m` for "contributions interrupted"; one with high
`digital_share` for "digital first". Always check such labels against the profile table and with people who know the
members.

**(2)** There's no single right answer. A useful test for any proposed action: could a branch manager start doing it
next month, and would you be able to tell whether it worked?

**(3)** Employer segmentation:

In [ ]:
employers = pd.read_csv("employer_profile.csv", dtype={"employer_no": "string"})

EMPLOYER_FEATURES = [
    "headcount_latest", "avg_wage_tzs", "on_time_rate_24m", "avg_days_late_24m", "unpaid_months_24m",
    "partial_payment_rate_24m", "headcount_cv_24m",
]
E = employers[EMPLOYER_FEATURES].copy()

# Log scale for headcount and wages, then fill any gaps with each column's median.
E["headcount_latest"] = np.log1p(E["headcount_latest"])
E["avg_wage_tzs"] = np.log(E["avg_wage_tzs"])
E = E.fillna(E.median())

In [ ]:
# Scale, then cluster into 4 groups.
Es = StandardScaler().fit_transform(E)

employer_kmeans = KMeans(n_clusters=4, n_init=20, random_state=SEED)
employer_kmeans.fit(Es)
employers["cluster"] = employer_kmeans.labels_

In [ ]:
# Profile each cluster in the original units, with arrears and the number of employers.
employer_profile = employers.groupby("cluster")[EMPLOYER_FEATURES + ["arrears_tzs"]].median()
employer_profile["employers"] = employers["cluster"].value_counts().sort_index()
employer_profile.round(2)

The cluster with low on-time rates, many unpaid months and high arrears is the compliance team's list.
Notice that it's defined by *behaviour*, not by size or sector, although you'll probably find some
sectors over-represented in it. Check with `pd.crosstab(employers["cluster"], employers["sector"])`.